In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
# 讀取訓練資料與測試資料
df_train = pd.read_csv('./cs-training.csv')
df_test = pd.read_csv('./cs-test.csv')

In [ ]:
# 檢視訓練資料的基本資訊
df_train.info()

In [ ]:
# (Optional) 刪除缺失值
# df_train.dropna(inplace=True)
# df_train.info()

In [ ]:
# 檢視測試資料的基本資訊
df_test.info()

In [ ]:
# 取得訓練資料的特徵與目標變數
X = df_train.iloc[:, 2:12]
y = df_train["SeriousDlqin2yrs"]

# 將資料分為訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

# 特徵縮放 (標準化)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 選擇模型
rf = RandomForestClassifier(random_state=0)

# 定義超參數綱格
param_grid = {
    'n_estimators': np.linspace(1, 500, 2, dtype=int),
    'max_depth': [None] + list(np.linspace(1, 500, 2, dtype=int)),
    'min_samples_split': np.linspace(2, 100, 2, dtype=int)
}

# 使用 GridSearchCV 進行超參數調整
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    n_jobs=2,
    verbose=3
)

# 執行 GridSearchCV
grid_search.fit(X_train_scaled, y_train)
print("Best Hyperparameters:", grid_search.best_params_)

# 使用最佳模型進行預測
y_pred = grid_search.predict(X_test_scaled)

# 計算 AUC-ROC
auc_roc = roc_auc_score(y_test, y_pred)
print("AUC-ROC:", auc_roc)

In [ ]:
# 取得 df_text 的特徵
X_ = df_test.iloc[:, 2:12]

# 標準化
X_scaled = scaler.transform(X_)

# 預測機率
y_pred_proba = grid_search.predict_proba(X_scaled)[:, 1]

# sampleEntry.csv 的欄位有 Id,Probability
submission = pd.DataFrame({
    'Id': df_test.index + 1, # Id 要從 index + 1 開始
    'Probability': y_pred_proba
})

# 儲存為 submission.csv
submission.to_csv('submission.csv', index=False)